# 02 - Preprocessing Check

**Goal:** trust the per-frame transform pipeline.

Visualize raw vs transformed frames and confirm tensor shape, dtype, and value range. Catches:
- wrong normalization,
- accidental color-channel reordering,
- destructive resizing.

Uses `smth2smth.shared.data.transforms.build_transforms`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from smth2smth.shared.data import (
    VideoFrameDataset,
    build_transforms,
    collect_video_samples,
)

import matplotlib.pyplot as plt
from PIL import Image
import torch

In [ ]:
TRAIN_DIR = REPO_ROOT / "data" / "train"
samples = collect_video_samples(TRAIN_DIR)
video_dir, label = samples[0]
frame_path = sorted(video_dir.glob("*.jpg"))[0]
raw = Image.open(frame_path).convert("RGB")

transform_eval = build_transforms(image_size=224, is_training=False)
tensor = transform_eval(raw)
print("tensor shape:", tuple(tensor.shape))
print("tensor dtype:", tensor.dtype)
print("tensor range:", float(tensor.min()), float(tensor.max()))

## Side-by-side: raw vs transformed (un-normalized for display)

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
display_tensor = (tensor * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(raw); axes[0].set_title("raw"); axes[0].axis("off")
axes[1].imshow(display_tensor.permute(1, 2, 0).numpy()); axes[1].set_title("transform_eval (un-normalized for display)"); axes[1].axis("off")
plt.tight_layout()

## Full dataset sample shape

In [ ]:
ds = VideoFrameDataset(
    root_dir=TRAIN_DIR,
    num_frames=8,
    transform=transform_eval,
)
video_tensor, label_tensor = ds[0]
print("video tensor (T, C, H, W):", tuple(video_tensor.shape))
print("label:", int(label_tensor))